## Week 4 (course alignment)

- Same workflow as `week4/day3.ipynb` / `week4/day5.ipynb`: load **`.env`** from the **repo root** (`llm_engineering/.env`) with `OPENAI_API_KEY`, optional `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`, `GROK_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`.
- **Run Jupyter with cwd = repo root** so `week4/system_info.py` imports work.
- Frontier models port Python → **Rust**; we **compile** with `rustc` and **assert** numeric parity with Python.
- **Benchmarks:** π (`day3`); `python_hard` (`day5`).
- `WEEK4_QUICK=1`: fast π iteration count. **`USE_LLM_FOR_RUST=0`**: skip API calls; use embedded Rust only. **`WEEK4_PORT_MODEL`**: override `SELECTED_MODEL` (default `gpt-5`).

## Algorithms

1. **π:** Leibniz-style partial sum — **Θ(N)**; LLM or hand-written Rust preserves evaluation order for float parity.
2. **LCG:** **Θ(1)** per sample.
3. **Max subarray:** Python **Θ(n²)**; embedded Rust uses **Kadane** **Θ(n)**. LLM output is tried first when enabled.

In [ ]:
# Repo root + week4 on sys.path (run notebook from llm_engineering, like week4/day5.ipynb)
from __future__ import annotations

import io
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "week4" / "system_info.py").is_file():
            return cand
    raise RuntimeError(
        "Cannot find week4/system_info.py — start Jupyter from the llm_engineering directory (repo root)."
    )


REPO_ROOT = _repo_root()
sys.path.insert(0, str(REPO_ROOT / "week4"))

_QUICK = os.environ.get("WEEK4_QUICK", "").strip().lower() in ("1", "true", "yes")
PI_ITERATIONS = 10_000 if _QUICK else 200_000_000
USE_LLM_FOR_RUST = os.environ.get("USE_LLM_FOR_RUST", "1").strip().lower() not in ("0", "false", "no")

In [ ]:
# .env from repo root — keys as in week4/day5.ipynb
load_dotenv(REPO_ROOT / ".env", override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (optional)")

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [ ]:
# Model registry (week4/day5.ipynb)
models = [
    "gpt-5",
    "claude-sonnet-4-5-20250929",
    "grok-4",
    "gemini-2.5-pro",
    "qwen2.5-coder",
    "deepseek-coder-v2",
    "gpt-oss:20b",
    "qwen/qwen3-coder-30b-a3b-instruct",
    "openai/gpt-oss-120b",
]

clients = {
    "gpt-5": openai,
    "claude-sonnet-4-5-20250929": anthropic,
    "grok-4": grok,
    "gemini-2.5-pro": gemini,
    "openai/gpt-oss-120b": groq,
    "qwen2.5-coder": ollama,
    "deepseek-coder-v2": ollama,
    "gpt-oss:20b": ollama,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
}

SELECTED_MODEL = os.environ.get("WEEK4_PORT_MODEL", "gpt-5")

In [ ]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

In [ ]:
# Optional: model-suggested rustc command line (week4/day5.ipynb)
_compile_advice = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

if openai_api_key and USE_LLM_FOR_RUST:
    _r = openai.chat.completions.create(
        model=models[0],
        messages=[{"role": "user", "content": _compile_advice}],
    )
    display(Markdown(_r.choices[0].message.content or ""))
else:
    print("Skipping compile advice (no OpenAI key or USE_LLM_FOR_RUST=0).")

In [ ]:
# Portable rustc + same -C flags as week4/day5.ipynb (without strip, for clearer errors)


def find_rustc() -> str:
    p = shutil.which("rustc")
    if p:
        return p
    candidate = Path.home() / ".cargo" / "bin" / "rustc"
    if candidate.is_file():
        return str(candidate)
    raise RuntimeError("rustc not found — install from https://rustup.rs/")


compile_command = [
    find_rustc(),
    "main.rs",
    "-C",
    "opt-level=3",
    "-C",
    "target-cpu=native",
    "-C",
    "codegen-units=1",
    "-C",
    "lto=fat",
    "-C",
    "panic=abort",
    "-o",
    "main",
]
compile_command

In [ ]:
# Helpers + LLM port (week4/day3/day5 pattern)


def run_python_captured(code: str) -> str:
    buf = io.StringIO()
    old = sys.stdout
    sys.stdout = buf
    g: dict = {"__builtins__": __builtins__}
    try:
        exec(code, g, g)
    finally:
        sys.stdout = old
    return buf.getvalue()


def rust_compile_and_run(source: str) -> str:
    exe_name = "main.exe" if sys.platform == "win32" else "main"
    with tempfile.TemporaryDirectory() as tmp:
        t = Path(tmp)
        (t / "main.rs").write_text(source, encoding="utf-8")
        subprocess.run(compile_command, check=True, text=True, capture_output=True, cwd=t)
        exe_path = t / exe_name
        run = subprocess.run(
            [str(exe_path)],
            check=True,
            text=True,
            capture_output=True,
            cwd=t,
        )
        return run.stdout


def parse_pi_result(stdout: str) -> float:
    m = re.search(r"Result:\s*([0-9]+\.[0-9]+)", stdout)
    if not m:
        raise ValueError(f"Could not parse Result line from:\n{stdout!r}")
    return float(m.group(1))


def parse_subarray_total(stdout: str) -> int:
    m = re.search(r"Total Maximum Subarray Sum \(20 runs\):\s*(-?[0-9]+)", stdout)
    if not m:
        raise ValueError(f"Could not parse total from:\n{stdout!r}")
    return int(m.group(1))


language = "Rust"
extension = "rs"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""


def user_prompt_for(python: str) -> str:
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{extension} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""


def messages_for(python: str):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)},
    ]


def port_rust(client: OpenAI, model: str, python: str) -> str:
    reasoning_effort = "high" if "gpt" in model else None
    kwargs: dict = {"model": model, "messages": messages_for(python)}
    if reasoning_effort is not None:
        kwargs["reasoning_effort"] = reasoning_effort
    response = client.chat.completions.create(**kwargs)
    reply = response.choices[0].message.content or ""
    reply = reply.replace("```rust", "").replace("```Rust", "").replace("```cpp", "").replace("```", "")
    return reply.strip()


def rust_source_llm_or_fallback(python_src: str, fallback: str, label: str) -> str:
    if not USE_LLM_FOR_RUST:
        print(f"{label}: USE_LLM_FOR_RUST=0 — embedded Rust.")
        return fallback
    if SELECTED_MODEL not in clients:
        print(f"{label}: bad SELECTED_MODEL — embedded Rust.")
        return fallback
    try:
        src = port_rust(clients[SELECTED_MODEL], SELECTED_MODEL, python_src)
        if "fn main" not in src:
            raise ValueError("model output missing fn main")
        return src
    except Exception as e:
        print(f"{label}: LLM failed ({e!r}) — embedded Rust.")
        return fallback

In [ ]:
# π — Python reference (week4/day3.ipynb)
pi_code = f"""import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= (1 / j)
        j = i * param1 + param2
        result += (1 / j)
    return result

start_time = time.time()
result = calculate({PI_ITERATIONS}, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

pi_py_out = run_python_captured(pi_code)
print(pi_py_out, end="")
pi_py_value = parse_pi_result(pi_py_out)

In [ ]:
# π — Rust via SELECTED_MODEL when USE_LLM_FOR_RUST, else embedded template
PI_RUST_TEMPLATE = '// Mirrors Python: same loop bounds and -= / += reciprocal sequence (f64).\nfn calculate(iterations: u64, param1: f64, param2: f64) -> f64 {{\n    let mut result = 1.0_f64;\n    for i in 1..=iterations {{\n        let mut j = i as f64 * param1 - param2;\n        result -= 1.0 / j;\n        j = i as f64 * param1 + param2;\n        result += 1.0 / j;\n    }}\n    result\n}}\n\nfn main() {{\n    const ITERATIONS: u64 = {iterations};\n    let start = std::time::Instant::now();\n    let result = calculate(ITERATIONS, 4.0, 1.0) * 4.0;\n    let elapsed = start.elapsed().as_secs_f64();\n    println!("Result: {{:.12}}", result);\n    println!("Execution Time: {{:.6}} seconds", elapsed);\n}}\n'
pi_fallback = PI_RUST_TEMPLATE.format(iterations=PI_ITERATIONS)
pi_rust_src = rust_source_llm_or_fallback(pi_code, pi_fallback, label='π')
pi_rs_out = rust_compile_and_run(pi_rust_src)
print(pi_rs_out, end="")
pi_rs_value = parse_pi_result(pi_rs_out)
assert abs(pi_py_value - pi_rs_value) < 1e-12, (pi_py_value, pi_rs_value)
print("π: Rust matches Python (12 dp).")

In [ ]:
python_hard = """
# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))

"""

hard_py_out = run_python_captured(python_hard)
print(hard_py_out, end="")
hard_py_total = parse_subarray_total(hard_py_out)

In [ ]:
# python_hard — Rust via LLM + Kadane fallback
HARD_RUST = '// LCG matches week4/day5 `python_hard`; max subarray via Kadane (O(n)), streaming O(1) extra memory.\nfn lcg_step(state: &mut u32) -> u32 {\n    const A: u64 = 1_664_525;\n    const C: u64 = 1_013_904_223;\n    let v = *state as u64;\n    let nxt = (A.wrapping_mul(v).wrapping_add(C)) % (1u64 << 32);\n    *state = nxt as u32;\n    *state\n}\n\nfn max_subarray_kadane_stream(mut inner: u32, n: usize, span: u32, min_val: i32) -> i64 {\n    let mut max_ending: i64 = 0;\n    let mut max_so_far: i64 = i64::MIN;\n    for _ in 0..n {\n        let raw = lcg_step(&mut inner);\n        let x = (raw % span) as i64 + min_val as i64;\n        max_ending = x.max(max_ending + x);\n        max_so_far = max_so_far.max(max_ending);\n    }\n    max_so_far\n}\n\nfn main() {\n    const N: usize = 10_000;\n    const INITIAL_SEED: u32 = 42;\n    const MIN_VAL: i32 = -10;\n    const MAX_VAL: i32 = 10;\n    let span = (MAX_VAL - MIN_VAL + 1) as u32;\n\n    let start = std::time::Instant::now();\n    let mut outer = INITIAL_SEED;\n    let mut total: i64 = 0;\n    for _ in 0..20 {\n        lcg_step(&mut outer);\n        let run_seed = outer;\n        total += max_subarray_kadane_stream(run_seed, N, span, MIN_VAL);\n    }\n    let elapsed = start.elapsed().as_secs_f64();\n    println!("Total Maximum Subarray Sum (20 runs): {}", total);\n    println!("Execution Time: {:.6} seconds", elapsed);\n}\n'
hard_rust_src = rust_source_llm_or_fallback(python_hard, HARD_RUST, label='python_hard')
hard_rs_out = rust_compile_and_run(hard_rust_src)
print(hard_rs_out, end="")
hard_rs_total = parse_subarray_total(hard_rs_out)
assert hard_py_total == hard_rs_total, (hard_py_total, hard_rs_total)
print("Subarray totals: Rust matches Python.")